# 02 — vLLM: Needle-in-a-Haystack Baseline

This notebook runs the same NIAH benchmark as notebook 01, but using
[vLLM](https://github.com/vllm-project/vllm) with Qwen3-8B **without
KV cache compression**.

vLLM uses PagedAttention for efficient memory management. This serves
as the quality and throughput baseline for comparison against
KeyDiffPress in notebook 03.

Results are saved to `results/vllm/` for comparison in notebook 03.

In [1]:
import os
os.environ["HF_TOKEN"] = "YOUR_TOKEN_HERE"

## Configuration

In [2]:
MODEL_NAME = "Qwen/Qwen3-8B"

NEEDLE_DEPTHS = [0, 25, 50, 75, 100]

MAX_CONTEXT_LENGTHS = [4096, 8192]

MAX_NEW_TOKENS = 64

In [3]:
import sys
import os

FORK_DIR = "/opt/app-root/src/vllm-fork"

if os.path.isdir(FORK_DIR) and os.listdir(FORK_DIR):
    sys.path.insert(0, FORK_DIR)
    import vllm
    print(f"Using FORK vLLM (version: {vllm.__version__})")
else:
    import vllm
    print(f"Using SYSTEM vLLM (version: {vllm.__version__})")

/opt/app-root/src/vllm-fork/vllm/__init__.py:7: RuntimeWarning: Failed to read commit hash:
No module named 'vllm._version'
  from .version import __version__, __version_tuple__  # isort:skip


Using FORK vLLM (version: dev)


In [4]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU detected.")

vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
allocated_gb = torch.cuda.memory_allocated() / 1e9
reserved_gb = torch.cuda.memory_reserved() / 1e9

print(f"GPU:        {torch.cuda.get_device_name(0)}")
print(f"VRAM:       {vram_gb:.1f} GB total")
print(f"Allocated:  {allocated_gb:.2f} GB")
print(f"Reserved:   {reserved_gb:.2f} GB")
print(f"Free:       {vram_gb - reserved_gb:.1f} GB (approx)")

if allocated_gb > 1.0:
    print(
        "\n⚠  GPU memory is not free — a model from another notebook may still be loaded.\n"
        "   Restart this kernel before proceeding."
    )

GPU:        NVIDIA A10G
VRAM:       23.7 GB total
Allocated:  0.00 GB
Reserved:   0.00 GB
Free:       23.7 GB (approx)


## 1. Load Model with vLLM

In [5]:
import torch
from vllm import LLM, SamplingParams

vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {torch.cuda.get_device_name(0)} — {vram_gb:.1f} GB VRAM")

llm_kwargs = {
    "model": MODEL_NAME,
    "dtype": "auto",
    "gpu_memory_utilization": 0.90,
    "max_model_len": max(MAX_CONTEXT_LENGTHS) + MAX_NEW_TOKENS + 256,
    "trust_remote_code": True,
}

if vram_gb < 20:
    print("Using 4-bit quantization (VRAM < 20 GB)")
    llm_kwargs["quantization"] = "bitsandbytes"
    llm_kwargs["load_format"] = "bitsandbytes"

llm = LLM(**llm_kwargs)

sampling_params = SamplingParams(
    temperature=0.0,
    max_tokens=MAX_NEW_TOKENS,
)

print("Model loaded.")

/opt/app-root/lib64/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


GPU: NVIDIA A10G — 23.7 GB VRAM
INFO 07-28 17:38:39 [utils.py:233] non-default args: {'trust_remote_code': True, 'max_model_len': 8512, 'disable_log_stats': True, 'model': 'Qwen/Qwen3-8B'}
INFO 07-28 17:38:39 [model.py:533] Resolved architecture: Qwen3ForCausalLM
INFO 07-28 17:38:39 [model.py:1582] Using max model len 8512
INFO 07-28 17:38:39 [scheduler.py:231] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 07-28 17:38:39 [vllm.py:754] Asynchronous scheduling is enabled.
WARNING 07-28 17:38:41 [system_utils.py:152] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized


/opt/app-root/src/vllm-fork/vllm/__init__.py:7: RuntimeWarning: Failed to read commit hash:
No module named 'vllm._version'
  from .version import __version__, __version_tuple__  # isort:skip


(EngineCore pid=2615) INFO 07-28 17:38:53 [core.py:103] Initializing a V1 LLM engine (vdev) with config: model='Qwen/Qwen3-8B', speculative_config=None, tokenizer='Qwen/Qwen3-8B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.bfloat16, max_seq_len=8512, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_config=ObservabilityConfig(show_hidden_metrics_for_version=None, otlp_traces_endpoint=None, collect_detailed_traces=None, kv_cach

Loading safetensors checkpoint shards:   0% Completed | 0/5 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  20% Completed | 1/5 [00:00<00:02,  1.91it/s]
Loading safetensors checkpoint shards:  40% Completed | 2/5 [00:01<00:01,  1.57it/s]
Loading safetensors checkpoint shards:  60% Completed | 3/5 [00:01<00:01,  1.49it/s]
Loading safetensors checkpoint shards:  80% Completed | 4/5 [00:02<00:00,  1.55it/s]
Loading safetensors checkpoint shards: 100% Completed | 5/5 [00:02<00:00,  1.92it/s]
Loading safetensors checkpoint shards: 100% Completed | 5/5 [00:02<00:00,  1.75it/s]
(EngineCore pid=2615) 


(EngineCore pid=2615) INFO 07-28 17:38:59 [default_loader.py:384] Loading weights took 2.92 seconds
(EngineCore pid=2615) INFO 07-28 17:39:00 [gpu_model_runner.py:4566] Model loading took 15.27 GiB memory and 3.884770 seconds
(EngineCore pid=2615) INFO 07-28 17:39:04 [backends.py:988] Using cache directory: /opt/app-root/src/.cache/vllm/torch_compile_cache/9186a7e81d/rank_0_0/backbone for vLLM's torch.compile
(EngineCore pid=2615) INFO 07-28 17:39:04 [backends.py:1048] Dynamo bytecode transform time: 3.76 s
(EngineCore pid=2615) INFO 07-28 17:39:06 [backends.py:284] Directly load the compiled graph(s) for compile range (1, 8192) from the cache, took 1.374 s
(EngineCore pid=2615) INFO 07-28 17:39:06 [monitor.py:48] torch.compile took 5.50 s in total
(EngineCore pid=2615) INFO 07-28 17:39:06 [decorators.py:296] Directly load AOT compilation from path /opt/app-root/src/.cache/vllm/torch_compile_cache/torch_aot_compile/fa0457bdc373d527415d87b54fc7b58b65e0029659d99e29c61223bb80091ba4/rank_0

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:04<00:00, 10.36it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:02<00:00, 13.85it/s]


(EngineCore pid=2615) INFO 07-28 17:39:18 [gpu_model_runner.py:5746] Graph capturing finished in 8 secs, took 0.51 GiB
(EngineCore pid=2615) INFO 07-28 17:39:18 [gpu_worker.py:617] CUDA graph pool memory: 0.51 GiB (actual), 0.52 GiB (estimated), difference: 0.01 GiB (1.5%).
(EngineCore pid=2615) INFO 07-28 17:39:18 [core.py:281] init engine (profile, create kv cache, warmup model) took 17.95 seconds
(EngineCore pid=2615) INFO 07-28 17:39:19 [vllm.py:754] Asynchronous scheduling is enabled.
INFO 07-28 17:39:19 [llm.py:391] Supported tasks: ['generate']
Model loaded.


## 2. Load Haystack Dataset

Paul Graham essays used as haystack filler, following the original
Needle-in-a-Haystack benchmark by [Kamradt (2023)](https://github.com/gkamradt/LLMTest_NeedleInAHaystack).

In [6]:
from datasets import load_dataset

haystack_ds = load_dataset("alessiodevoto/paul_graham_essays", split="test")
haystack_df = haystack_ds.to_pandas()

print(f"Haystack dataset loaded: {len(haystack_df)} rows")
print(f"Columns: {list(haystack_df.columns)}")
print(f"Needle: {haystack_df['needle'].iloc[0][:100]}...")
print(f"Question: {haystack_df['question'].iloc[0]}")

Haystack dataset loaded: 1 rows
Columns: ['context', 'needle', 'question', 'answer_prefix', 'max_new_tokens']
Needle: 

Remember, the best thing to do in San Francisco is eat a sandwich and sit in Dolores Park on a sun...
Question: 

Question: Based on the content of the book, what is the best thing to do in San Francisco?


## 3. Prepare Prompts

For each (max_context_length, needle_depth) combination, we build the
haystack with needle inserted, then apply the model's chat template
to produce the final prompt for vLLM.

In [7]:
from transformers import AutoTokenizer
from eval_utils import insert_needle_in_haystack

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

prompt_configs = []

for max_ctx_len in MAX_CONTEXT_LENGTHS:
    niah_df = insert_needle_in_haystack(
        haystack_df, tokenizer, max_ctx_len, NEEDLE_DEPTHS,
    )

    for idx, row in niah_df.iterrows():
        user_msg = row["context"] + "\n\n" + row["question"]
        messages = [{"role": "user", "content": user_msg}]
        prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True,
        )
        if row["answer_prefix"]:
            prompt += row["answer_prefix"]

        prompt_configs.append({
            "prompt": prompt,
            "max_context_length": max_ctx_len,
            "needle_depth": row["needle_depth"],
            "needle": row["needle"],
            "question": row["question"],
        })

print(f"Prepared {len(prompt_configs)} prompts")

Token indices sequence length is longer than the specified maximum sequence length for this model (699555 > 131072). Running this sequence through the model will result in indexing errors


Prepared 10 prompts


## 4. Run Batch Inference

vLLM processes all prompts in a single batch via its internal scheduler.

In [8]:
import time
import json
from eval_utils import calculate_niah_metrics, rouge_l_f_scores

def get_gpu_memory_used_gb() -> float:
    """Actual GPU memory used, measured at the CUDA driver level.
    Works for vLLM which manages its own memory pool outside PyTorch."""
    free, total = torch.cuda.mem_get_info()
    return (total - free) / 1e9

prompts = [pc["prompt"] for pc in prompt_configs]

mem_before = get_gpu_memory_used_gb()
start = time.perf_counter()

outputs = llm.generate(prompts, sampling_params)

batch_elapsed = time.perf_counter() - start
mem_after = get_gpu_memory_used_gb()
peak_mem = max(mem_before, mem_after)

all_results = []
for i, output in enumerate(outputs):
    predicted_answer = output.outputs[0].text.strip()
    pc = prompt_configs[i]

    result = {
        "framework": "vllm",
        "press": "none",
        "compression_ratio": 0.0,
        "max_context_length": pc["max_context_length"],
        "needle_depth": pc["needle_depth"],
        "needle": pc["needle"],
        "question": pc["question"],
        "predicted_answer": predicted_answer,
        "elapsed_sec": round(batch_elapsed / len(prompts), 3),
        "peak_gpu_mem_gb": round(peak_mem, 3),
        "task": "needle_in_haystack",
    }
    all_results.append(result)

total_gen_tokens = sum(len(o.outputs[0].token_ids) for o in outputs)
throughput = total_gen_tokens / batch_elapsed if batch_elapsed > 0 else 0

print(f"Batch done in {batch_elapsed:.1f}s — {throughput:.1f} tok/s — peak mem={peak_mem:.2f} GB")
print(f"Total results: {len(all_results)}")

Processed prompts: 100%|██████████| 10/10 [00:13<00:00,  1.31s/it, est. speed input: 4600.12 toks/s, output: 34.15 toks/s]

Batch done in 13.3s — 33.7 tok/s — peak mem=23.21 GB
Total results: 10


## 5. Score Predictions

In [9]:
import pandas as pd

df = pd.DataFrame(all_results)

metrics = calculate_niah_metrics(df)
df["rouge_l_f"] = rouge_l_f_scores(metrics)

summary = (
    df.groupby(["max_context_length", "needle_depth"])
    .agg(
        rouge_l=("rouge_l_f", "mean"),
        mean_time=("elapsed_sec", "mean"),
        peak_mem=("peak_gpu_mem_gb", "max"),
    )
    .round(4)
)

print(summary.to_string())

                                 rouge_l  mean_time  peak_mem
max_context_length needle_depth                              
4096               0              0.6462      1.331    23.215
                   25             0.7119      1.331    23.215
                   50             0.7119      1.331    23.215
                   75             0.7119      1.331    23.215
                   100            0.7119      1.331    23.215
8192               0              0.7119      1.331    23.215
                   25             0.7119      1.331    23.215
                   50             0.7119      1.331    23.215
                   75             0.7119      1.331    23.215
                   100            0.7119      1.331    23.215


## 6. Save Results

In [10]:
import os

os.makedirs("results/vllm", exist_ok=True)

predictions_path = "results/vllm/predictions.csv"
df.to_csv(predictions_path, index=False)
print(f"Saved predictions to {predictions_path}")

metrics_path = "results/vllm/metrics.json"
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)
print(f"Saved metrics to {metrics_path}")

Saved predictions to results/vllm/predictions.csv
Saved metrics to results/vllm/metrics.json
